# 환각을 평가하고 스스로 교정하는 RAG (Self-RAG / CRAG)

고급 RAG 는 검색·생성의 각 단계에서 **여러 평가(grade)** 를 거쳐 답변 품질을 보장한다. 이 노트북은 세 가지 평가를 결합한다:

1. **문서 관련성 평가** — 검색 문서가 질문과 맞나? 아니면 질문 재작성(transform_query)
2. **환각 평가** — 생성된 답변이 검색 문서에 **근거**하나? 아니면 재생성
3. **답변 해결성 평가** — 답변이 질문을 실제로 **해결**하나? 아니면 질문 재작성

또한 내부 문서로 답할 수 없으면 **웹검색 에이전트** 로 폴백한다.

```
START → agent ─(검색필요)→ retrieve → grade_documents ─(관련)→ generate → [환각? 해결?]
          │(불필요)                         │(무관)              │useful → END
          ▼                          transform_query ◀──────────┤not useful
    web_search_agent → END               ▲ └─(재검색)→ retrieve  └not supported → generate(재생성)
```

> `OPENAI_API_KEY`, `TAVILY_API_KEY` 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("USER_AGENT", "ai-agent-study")
for k in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(k), f"{k} 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 0. Retriever 준비
[basics 복습] 공개 웹 문서를 로드·청킹·임베딩해 retriever 구성 (01 참고).

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.tools.retriever import create_retriever_tool

pages = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/").load()
docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)
vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_agent_docs",
    "Search and return information about LLM-based AI agents (planning, memory, tool use, reflection).",
)
print("retriever 준비 완료")

## Graph State
[basics 복습] `MessagesState` 상속 + 질문/생성/문서 필드 추가.

In [ ]:
from langgraph.graph import MessagesState
from langchain_openai import ChatOpenAI

class State(MessagesState):
    question: str     # 현재 질문 (재작성되면 갱신)
    generation: str   # LLM 생성 답변
    document: str     # 검색된 문서 내용

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Step 1. 노드들

### 1) agent — 검색 도구 호출 여부 판단
[basics 복습] bind_tools 로 검색이 필요한지 판단. 검색 불필요하면 (tool_calls 없음) 웹검색으로 폴백된다.

In [ ]:
def agent(state: State):
    print("##### AGENT #####")
    messages = state["messages"]
    llm_with_tools = llm.bind_tools([retriever_tool])
    response = llm_with_tools.invoke(messages)
    return {"messages": [response], "question": messages[0].content}

### 2) retrieve — 문서 검색

In [ ]:
def retrieve(state: State):
    print("##### RETRIEVE #####")
    question = state["question"]
    document = retriever.invoke(question)
    return {"document": document[0].page_content, "question": question}

### 3) grade_documents — 문서 관련성 평가
[basics 복습] 구조화 출력으로 yes/no. 무관하면 document 를 비워 이후 재작성으로 보낸다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Grade(BaseModel):
    """관련성 이진 점수"""
    binary_score: str = Field(description="관련성 'yes' 또는 'no'")

def grade_documents(state: State):
    print("##### CHECK RELEVANCE #####")
    grader = llm.with_structured_output(Grade)
    grader_prompt = ChatPromptTemplate.from_template(
        """You are a grader assessing relevance of a retrieved document to a question.
        Document:\n{context}\n\nQuestion: {question}
        If it contains related keywords or meaning, grade 'yes'. Loose test to filter wrong retrievals.
        Give 'yes' or 'no'."""
    )
    chain = grader_prompt | grader
    question, document = state["question"], state["document"]
    grade = chain.invoke({"question": question, "context": document}).binary_score
    if grade == "yes":
        print("---DOC RELEVANT---")
        return {"document": document, "question": question}
    print("---DOC NOT RELEVANT---")
    return {"document": "", "question": question}   # 비워서 재작성 유도

### 4) generate — 답변 생성
[basics 복습] 검색 문서를 근거로 RAG 프롬프트로 답변 (프롬프트 인라인).

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for QA. Use the retrieved context to answer. "
     "If you don't know, say so. Three sentences max.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

def generate(state: State):
    print("##### GENERATE #####")
    question, document = state["question"], state["document"]
    response = llm.invoke(RAG_PROMPT.format_messages(context=document, question=question))
    return {"question": question, "document": document,
            "generation": response.content, "messages": [response]}

### 5) transform_query — 질문 재작성
검색에 더 적합하도록 질문을 다듬는다 (사람 개입 없이 LLM 이 자동으로).

In [ ]:
def transform_query(state: State):
    print("##### TRANSFORM QUERY #####")
    question = state["question"]
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ("system", "You rewrite a question to be better optimized for vectorstore retrieval. "
                    "Reason about the underlying semantic intent."),
        ("user", "Initial question:\n{question}\nFormulate an improved question in Korean."),
    ])
    better = (rewrite_prompt | llm).invoke({"question": question})
    return {"question": better.content, "messages": [better]}

### 6) web_search_agent — 내부 문서로 불충분할 때 웹 폴백
[basics 복습] `create_react_agent` + Tavily 로 외부 웹검색 에이전트를 만든다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

web_search_agent = create_react_agent(llm, tools=[TavilySearchResults(max_results=3)])

## Step 2. 평가용 엣지(라우터)

### 1) decide_to_generate — 문서 있으면 생성, 없으면 질문 재작성

In [ ]:
def decide_to_generate(state: State):
    print("##### ASSESS DOCUMENTS #####")
    if state["document"] == "":
        print("---NOT RELEVANT → TRANSFORM QUERY---")
        return "transform_query"
    print("---RELEVANT → GENERATE---")
    return "generate"

### 2) 환각 평가 + 답변 해결성 평가

두 평가기를 만든다:
- **환각 평가**: 답변이 문서(facts)에 근거하나? (`GradeHallucinations`)
- **해결성 평가**: 답변이 질문을 해결하나? (`GradeAnswer`)

결과 분기:
- 환각 있음 → `not supported` (답변 재생성)
- 환각 없음 + 질문 해결 → `useful` (종료)
- 환각 없음 + 질문 미해결 → `not useful` (질문 재작성)

In [ ]:
class GradeAnswer(BaseModel):
    """답변이 질문을 해결하는지"""
    binary_score: str = Field(description="'yes' 또는 'no'")

class GradeHallucinations(BaseModel):
    """답변이 사실에 근거하는지"""
    binary_score: str = Field(description="근거함 'yes' 또는 'no'")

answer_grader = (
    ChatPromptTemplate.from_messages([
        ("system", "You assess whether an answer resolves a question. Ambiguous → no. Give 'yes' or 'no'."),
        ("user", "Question:\n{question}\n\nGeneration: {generation}"),
    ]) | llm.with_structured_output(GradeAnswer)
)

hallucination_grader = (
    ChatPromptTemplate.from_messages([
        ("system", "You assess whether a generation is grounded in the retrieved facts. Give 'yes' or 'no'."),
        ("user", "Facts:\n{document}\n\nGeneration: {generation}"),
    ]) | llm.with_structured_output(GradeHallucinations)
)

In [ ]:
def grade_generation_v_documents_and_question(state: State):
    print("##### CHECK HALLUCINATIONS #####")
    question, document, generation = state["question"], state["document"], state["generation"]

    grounded = hallucination_grader.invoke(
        {"document": document, "generation": generation}
    ).binary_score
    if grounded != "yes":
        print("---NOT GROUNDED → RE-GENERATE---")
        return "not supported"

    print("---GROUNDED → CHECK ANSWER---")
    resolves = answer_grader.invoke(
        {"question": question, "generation": generation}
    ).binary_score
    if resolves == "yes":
        print("---ADDRESSES QUESTION → USEFUL---")
        return "useful"
    print("---DOES NOT ADDRESS → TRANSFORM QUERY---")
    return "not useful"

## Step 3. 그래프 컴파일

- `agent` → 검색 필요하면 `retrieve`, 아니면 `web_search_agent`(웹 폴백)
- `retrieve` → `grade_documents` → (관련 `generate` / 무관 `transform_query`)
- `generate` → 환각·해결성 평가 → (`useful`=END / `not supported`=재생성 / `not useful`=재작성)
- `transform_query` → `retrieve` (재검색)

In [ ]:
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import tools_condition

graph_builder = StateGraph(State)
graph_builder.add_node("agent", agent)
graph_builder.add_node("web_search_agent", web_search_agent)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade_documents", grade_documents)
graph_builder.add_node("generate", generate)
graph_builder.add_node("transform_query", transform_query)

graph_builder.add_edge(START, "agent")
# 검색 도구 호출 있으면 retrieve, 없으면 웹검색 폴백
graph_builder.add_conditional_edges(
    "agent", tools_condition, {"tools": "retrieve", END: "web_search_agent"}
)
graph_builder.add_edge("web_search_agent", END)

graph_builder.add_edge("retrieve", "grade_documents")
graph_builder.add_conditional_edges(
    "grade_documents", decide_to_generate,
    {"transform_query": "transform_query", "generate": "generate"},
)
graph_builder.add_edge("transform_query", "retrieve")
graph_builder.add_conditional_edges(
    "generate", grade_generation_v_documents_and_question,
    {"not supported": "generate", "useful": END, "not useful": "transform_query"},
)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트

`recursion_limit` 초과를 대비해 `GraphRecursionError` 를 잡는다.

### case 1. 내부 문서로 답할 수 없는 질문 → 웹검색 폴백

In [ ]:
from langgraph.errors import GraphRecursionError

inputs = {"messages": "2024년 노벨문학상 수상자는 누구인가요?"}
try:
    response = graph.invoke(inputs)
    response["messages"][-1].pretty_print()
except GraphRecursionError:
    print("Recursion Error")

### case 2. 내부 문서로 답할 수 있는 질문 → RAG

In [ ]:
inputs = {"messages": "Explain the planning component of an LLM agent."}
try:
    response = graph.invoke(inputs)
    response["messages"][-1].pretty_print()
except GraphRecursionError:
    print("Recursion Error")

### case 3. 부실한 질문 → 재작성 후 재검색

In [ ]:
inputs = {"messages": "agent agent agent"}
try:
    response = graph.invoke(inputs, config={"recursion_limit": 20})
    response["messages"][-1].pretty_print()
except GraphRecursionError:
    print("Recursion Error")

## 정리

- **3중 평가**로 RAG 품질을 자가 교정:
  - 문서 관련성 → 무관하면 질문 재작성
  - 환각 평가 → 근거 없으면 답변 재생성
  - 해결성 평가 → 질문 미해결이면 질문 재작성
- 내부 문서 부족 시 **웹검색 에이전트** 폴백
- 평가는 모두 **구조화 출력(yes/no)** + 조건부 엣지 분기로 구현

다음: 부족하면 처음부터 **웹에서 보강** 하는 RAG (corrective RAG).